# 04 - Feature Engineering + XGBoost Optimierung

**Project:** Diabetes Prediction ML  
**Seminar:** Advanced Applied Data Science - Goethe Universitaet Frankfurt  
**Dataset:** CDC BRFSS 2015 - Diabetes Health Indicators (UCI #891)  
**CRISP-DM Phase:** 4 - Modeling (Feature Engineering + Best XGBoost)

---

**Ziel:** Durch Feature Engineering klinisch sinnvolle Merkmale konstruieren und damit
das bestmoegliche XGBoost-Modell trainieren. Als Referenz dient das getunete XGBoost
aus Notebook 03 (ohne Feature Engineering).

**Grundlage:** Erkenntnisse aus Notebook 02 (Data Exploration):
- Top-Korrelationen mit Zielvariable: GenHlth > HighBP > BMI > Age > HighChol > DiffWalk
- MentHlth + PhysHlth: stark zero-inflated (~75% = 0) -> binaeres Flag robuster als Summe
- HighBP x HighChol: moderate Inter-Feature-Korrelation -> Interaktion klinisch begruendet
- BMI: rechtsschiefe Verteilung, kontinuierlich -> WHO-Klassen als Kategorisierung sinnvoll

**Vorgehen:**
1. Feature Engineering - 9 neue Features
2. XGBoost Referenz (kein FE)
3. XGBoost default + FE (isolierter FE-Effekt)
4. XGBoost tuned + FE (erweiterter Suchraum)
5. Feature Importance + Finaler Vergleich


## Setup

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    average_precision_score, roc_auc_score, accuracy_score,
    classification_report, confusion_matrix, precision_recall_curve
)
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)

PLOTS_DIR = os.path.join("..", "results", "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

sns.set_style("whitegrid")
sns.set_palette("Set2")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

LABEL_MAP    = {0: "No Diabetes", 1: "Prediabetes/Diabetes"}
CLASS_COLORS = {0: "#4CAF50", 1: "#E53935"}

print("Libraries loaded.  Seed =", SEED)


## 1 - Dataset laden

In [ ]:
dataset = fetch_ucirepo(id=891)
X_raw = dataset.data.features.astype(int).copy()
y     = dataset.data.targets.squeeze().astype(int)

print(f"Features : {X_raw.shape[0]:,} samples  x  {X_raw.shape[1]} columns")
print(f"Target   : {y.name}")
print()
for k, v in y.value_counts().sort_index().items():
    print(f"  {LABEL_MAP[k]:28s}: {v:,}  ({v / len(y) * 100:.1f}%)")

baseline_rate    = y.mean()
scale_pos_weight = int((y == 0).sum() / (y == 1).sum())

print(f"\nNo-skill PR-AUC  : {baseline_rate:.4f}")
print(f"scale_pos_weight : {scale_pos_weight}")


## 2 - Feature Engineering

Alle 9 neuen Features basieren auf klinischem Domainwissen und den Erkenntnissen
aus der Datenexploration (Notebook 02).

| Feature | Typ | Begruendung aus Notebook 02 |
|---------|-----|-----------------------------|
| `BMI_cat` | Ordinal 0-5 | BMI rechtsschiefe kontinuierliche Variable; WHO-Klassen klinisch standard |
| `Risk_score` | Count 0-9 | Buendelt Top-Korrelatioren: HighBP, HighChol, DiffWalk, Stroke |
| `Unhealthy_lifestyle` | Count 0-3 | Rauchen + Alkohol + keine Bewegung |
| `Any_poor_health` | Binary | MentHlth/PhysHlth zero-inflated (75%=0): binaeres Flag robuster |
| `Health_burden` | Numeric 0-60 | Summe schlechter Tage (ergaenzend zum binaeren Flag) |
| `BP_Chol_both` | Binary | HighBP x HighChol: moderate Inter-Feature-Korrelation, klinisch relevant |
| `Age_BMI_cat` | Numeric | Age (Rang 4) x BMI_cat (Rang 3): beide starke Einzelpraediktoren |
| `Age_GenHlth` | Numeric | Age x GenHlth (Rang 1): staerkste Einzelkombination |
| `Protective_behaviors` | Count 0-3 | PhysActivity + Fruits + Veggies (schwach individuell, staerker kombiniert) |


In [ ]:
def engineer_features(X):
    Xf = X.copy()

    # 1 - BMI WHO obesity classes (BMI is continuous actual BMI value, confirmed Notebook 02)
    Xf["BMI_cat"] = pd.cut(
        Xf["BMI"],
        bins=[0, 18.5, 25, 30, 35, 40, 999],
        labels=[0, 1, 2, 3, 4, 5],
        right=False
    ).astype(int)

    # 2 - Composite clinical risk score (top correlators from Notebook 02)
    Xf["Risk_score"] = (
        Xf["HighBP"] +
        Xf["HighChol"] +
        Xf["Smoker"] +
        Xf["Stroke"] +
        Xf["HeartDiseaseorAttack"] +
        Xf["DiffWalk"] +
        (1 - Xf["PhysActivity"]) +
        (1 - Xf["Fruits"]) +
        (1 - Xf["Veggies"])
    )

    # 3 - Unhealthy lifestyle count
    Xf["Unhealthy_lifestyle"] = (
        Xf["Smoker"] +
        Xf["HvyAlcoholConsump"] +
        (1 - Xf["PhysActivity"])
    )

    # 4 - Binary flag for any poor health days
    # Notebook 02: MentHlth and PhysHlth are zero-inflated (~75% zeros)
    # Binary flag is more robust than raw sum for tree-based models
    Xf["Any_poor_health"] = ((Xf["MentHlth"] + Xf["PhysHlth"]) > 0).astype(int)

    # 5 - Total poor health days (kept as complement to binary flag)
    Xf["Health_burden"] = Xf["MentHlth"] + Xf["PhysHlth"]

    # 6 - Comorbidity: hypertension AND high cholesterol simultaneously
    # Notebook 02: HighBP x HighChol show moderate inter-feature correlation
    Xf["BP_Chol_both"] = (Xf["HighBP"] & Xf["HighChol"]).astype(int)

    # 7 - Age x BMI interaction (Age rank 4, BMI rank 3 in correlation with target)
    Xf["Age_BMI_cat"] = Xf["Age"] * Xf["BMI_cat"]

    # 8 - Age x GenHlth interaction (GenHlth rank 1, Age rank 4)
    Xf["Age_GenHlth"] = Xf["Age"] * Xf["GenHlth"]

    # 9 - Protective behaviors count
    Xf["Protective_behaviors"] = (
        Xf["PhysActivity"] +
        Xf["Fruits"] +
        Xf["Veggies"]
    )

    return Xf


X_fe = engineer_features(X_raw)

new_features = [
    "BMI_cat", "Risk_score", "Unhealthy_lifestyle", "Any_poor_health",
    "Health_burden", "BP_Chol_both", "Age_BMI_cat", "Age_GenHlth",
    "Protective_behaviors"
]

print(f"Original features : {X_raw.shape[1]}")
print(f"After FE          : {X_fe.shape[1]}  (+{X_fe.shape[1] - X_raw.shape[1]} neue Features)")
print()
print(X_fe[new_features].describe().round(2).to_string())


In [ ]:
# Verteilung neuer Features nach Klasse
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, feat in enumerate(new_features):
    ax = axes[i]
    for cls, color, label in [(0, "#4CAF50", "No Diabetes"), (1, "#E53935", "Prediabetes/Diabetes")]:
        vals = X_fe.loc[y == cls, feat]
        if vals.nunique() <= 12:
            counts = vals.value_counts(normalize=True).sort_index()
            ax.bar([str(v) for v in counts.index], counts.values,
                   alpha=0.65, label=label, color=color, width=0.4)
        else:
            ax.hist(vals, bins=25, alpha=0.55, density=True, label=label, color=color)
    ax.set_title(feat, fontweight="bold", fontsize=10)
    ax.legend(fontsize=7)
    sns.despine(ax=ax)

fig.suptitle("Neue Features - Verteilung nach Zielklasse", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "21_fe_distributions.png"), bbox_inches="tight")
plt.show()


In [ ]:
# Korrelation neuer Features mit Zielvariable
corr_new = X_fe[new_features].corrwith(y).abs().sort_values(ascending=False)
corr_orig = X_raw.corrwith(y).abs().sort_values(ascending=False)

print("Korrelation NEUER Features mit Zielvariable (|r|):")
print(corr_new.round(4).to_string())
print("\nTop 5 ORIGINALE Features (zum Vergleich):")
print(corr_orig.head(5).round(4).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#E53935" if v > 0.15 else "#4E79A7" for v in corr_new.values]
ax.barh(corr_new.index[::-1], corr_new.values[::-1],
        color=colors[::-1], alpha=0.85, edgecolor="white")
ax.axvline(0.15, color="gray", linestyle="--", lw=1.2, label="0.15 threshold")
ax.set_xlabel("|Pearson r| mit Diabetes_binary")
ax.set_title("Neue Features - Korrelation mit Zielvariable", fontweight="bold")
ax.legend(fontsize=9)
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "22_fe_correlations.png"), bbox_inches="tight")
plt.show()


## 3 - Cross-Validation Framework

Identisch mit Notebook 03: Stratified 5-Fold CV, Seed=42.


In [ ]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


def evaluate_fold(y_true, y_pred, y_prob):
    return {
        "pr_auc"   : average_precision_score(y_true, y_prob),
        "roc_auc"  : roc_auc_score(y_true, y_prob),
        "accuracy" : accuracy_score(y_true, y_pred),
        "report"   : classification_report(
            y_true, y_pred,
            target_names=list(LABEL_MAP.values()),
            output_dict=True
        ),
        "confusion" : confusion_matrix(y_true, y_pred),
        "y_true"    : np.array(y_true),
        "y_prob"    : np.array(y_prob),
    }


def run_cv(estimator, X, y, cv, label="Model"):
    print(f"\n[{label}]")
    fold_results = []
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        estimator.fit(X_tr, y_tr)
        y_pred = estimator.predict(X_val)
        y_prob = estimator.predict_proba(X_val)[:, 1]
        res = evaluate_fold(y_val, y_pred, y_prob)
        res["fold"] = fold
        fold_results.append(res)
        print(f"  Fold {fold}:  PR-AUC={res['pr_auc']:.4f}  "
              f"ROC-AUC={res['roc_auc']:.4f}  Acc={res['accuracy']:.4f}")
    return fold_results


def print_summary(results, label):
    pr  = [r["pr_auc"]  for r in results]
    roc = [r["roc_auc"] for r in results]
    acc = [r["accuracy"] for r in results]
    print(f"\n=== {label} - {N_SPLITS}-Fold Summary ===")
    print(f"  PR-AUC  (primary) : {np.mean(pr):.4f}  +/-  {np.std(pr):.4f}")
    print(f"  ROC-AUC           : {np.mean(roc):.4f}  +/-  {np.std(roc):.4f}")
    print(f"  Accuracy          : {np.mean(acc):.4f}  +/-  {np.std(acc):.4f}")
    classes = list(LABEL_MAP.values()) + ["macro avg", "weighted avg"]
    rows = {}
    for cls in classes:
        if cls in results[0]["report"] and isinstance(results[0]["report"][cls], dict):
            rows[cls] = {m: np.mean([r["report"][cls][m] for r in results])
                         for m in ["precision", "recall", "f1-score", "support"]}
    report_df = pd.DataFrame(rows).T
    report_df["support"] = report_df["support"].astype(int)
    print("\n  Classification Report (mean across folds):")
    print(report_df.round(4).to_string())
    return np.mean(pr), np.mean(roc), np.mean(acc)


def get_metric(results, key):
    if key in ("pr_auc", "roc_auc", "accuracy"):
        vals = [r[key] for r in results]
    elif key == "recall":
        vals = [r["report"]["Prediabetes/Diabetes"]["recall"] for r in results]
    elif key == "f1":
        vals = [r["report"]["Prediabetes/Diabetes"]["f1-score"] for r in results]
    return np.mean(vals), np.std(vals)


print(f"Stratified {N_SPLITS}-Fold CV  |  shuffle=True  |  seed={SEED}")


## 4 - Referenz: XGBoost getunt ohne Feature Engineering

Direkter Vergleichspunkt: XGBoost auf den originalen 21 Features, 25 Iterationen x 3-Fold.


In [ ]:
param_dist_ref = {
    "max_depth"        : [3, 4, 5, 6, 7],
    "learning_rate"    : [0.01, 0.05, 0.1, 0.2],
    "n_estimators"     : [100, 200, 300, 500],
    "min_child_weight" : [1, 3, 5],
    "subsample"        : [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree" : [0.7, 0.8, 0.9, 1.0],
    "gamma"            : [0, 0.1, 0.2],
}

search_ref = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=SEED,
                  eval_metric="aucpr", verbosity=0),
    param_distributions=param_dist_ref,
    n_iter=25, scoring="average_precision",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    random_state=SEED, n_jobs=-1, verbose=0
)

print("Tuning XGBoost (kein FE) - 25 iter x 3-fold ...")
search_ref.fit(X_raw, y)
print(f"Best params : {search_ref.best_params_}")
print(f"Best PR-AUC : {search_ref.best_score_:.4f}  (inner 3-fold CV)")

xgb_ref_tuned = XGBClassifier(
    **search_ref.best_params_,
    scale_pos_weight=scale_pos_weight,
    random_state=SEED, eval_metric="aucpr", verbosity=0
)

results_ref = run_cv(xgb_ref_tuned, X_raw, y, skf, label="XGBoost tuned (kein FE)")
pr_ref, roc_ref, _ = print_summary(results_ref, "XGBoost tuned (kein FE)")


## 5 - XGBoost mit Feature Engineering - Default-Parameter

Isolierter Effekt des Feature Engineerings:
Default-Parameter, 21 + 9 = 30 Features.


In [ ]:
results_fe_default = run_cv(
    XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=SEED,
                  eval_metric="aucpr", verbosity=0),
    X_fe, y, skf, label="XGBoost default + FE"
)
pr_fe_default, roc_fe_default, _ = print_summary(results_fe_default, "XGBoost default + FE")


## 6 - XGBoost mit Feature Engineering - Erweitertes Tuning

Gegenueber Notebook 03 erweiterter Suchraum mit L1/L2-Regularisierung und colsample_bylevel.
50 Iterationen x 3-Fold = 150 Modell-Fits.

| Parameter | Suchraum |
|-----------|----------|
| `max_depth` | 3-8 |
| `learning_rate` | 0.005-0.2 |
| `n_estimators` | 200-1000 |
| `min_child_weight` | 1, 3, 5, 7 |
| `subsample` | 0.6-1.0 |
| `colsample_bytree` | 0.6-1.0 |
| `colsample_bylevel` | 0.6-1.0 |
| `gamma` | 0, 0.1, 0.2, 0.5 |
| `reg_alpha` | 0, 0.01, 0.1, 1.0 |
| `reg_lambda` | 0.5, 1.0, 2.0, 5.0 |


In [ ]:
param_dist_fe = {
    "max_depth"         : [3, 4, 5, 6, 7, 8],
    "learning_rate"     : [0.005, 0.01, 0.05, 0.1, 0.2],
    "n_estimators"      : [200, 300, 500, 700, 1000],
    "min_child_weight"  : [1, 3, 5, 7],
    "subsample"         : [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree"  : [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bylevel" : [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma"             : [0, 0.1, 0.2, 0.5],
    "reg_alpha"         : [0, 0.01, 0.1, 1.0],
    "reg_lambda"        : [0.5, 1.0, 2.0, 5.0],
}

search_fe = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=SEED,
                  eval_metric="aucpr", verbosity=0),
    param_distributions=param_dist_fe,
    n_iter=50, scoring="average_precision",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    random_state=SEED, n_jobs=-1, verbose=1
)

print("Tuning XGBoost + FE - 50 iter x 3-fold (150 fits) ...")
search_fe.fit(X_fe, y)
best_params_fe = search_fe.best_params_

print(f"\nBest params : {best_params_fe}")
print(f"Best PR-AUC : {search_fe.best_score_:.4f}  (inner 3-fold CV)")


In [ ]:
xgb_fe_tuned = XGBClassifier(
    **best_params_fe,
    scale_pos_weight=scale_pos_weight,
    random_state=SEED, eval_metric="aucpr", verbosity=0
)

results_fe_tuned = run_cv(xgb_fe_tuned, X_fe, y, skf, label="XGBoost tuned + FE")
pr_fe_tuned, roc_fe_tuned, _ = print_summary(results_fe_tuned, "XGBoost tuned + FE")


## 7 - Feature Importance

XGBoost gain-basierte Importance: durchschnittlicher Informationsgewinn pro Split.
Rot = neue FE-Features, Blau = originale Features.


In [ ]:
xgb_fe_tuned.fit(X_fe, y)
importances = pd.Series(
    xgb_fe_tuned.feature_importances_,
    index=X_fe.columns
).sort_values(ascending=False)

top_n   = 20
top_imp = importances.head(top_n)
colors  = ["#E53935" if f in new_features else "#4E79A7" for f in top_imp.index]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_imp.index[::-1], top_imp.values[::-1],
        color=colors[::-1], alpha=0.88, edgecolor="white")
ax.set_xlabel("Feature Importance (gain)", fontsize=11)
ax.set_title(
    f"Top {top_n} Feature Importances - XGBoost tuned + FE\n"
    "(Rot = neue Features, Blau = originale Features)",
    fontweight="bold"
)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#E53935", alpha=0.88, label="Neue Features (FE)"),
    Patch(facecolor="#4E79A7", alpha=0.88, label="Originale Features")
], fontsize=10)
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "23_fe_importance.png"), bbox_inches="tight")
plt.show()
print("\nTop 10 Features:")
print(top_imp.head(10).round(4).to_string())


## 8 - Finaler Modellvergleich

In [ ]:
model_results = {
    "XGBoost\ntuned\n(kein FE)"  : results_ref,
    "XGBoost\ndefault\n+ FE"     : results_fe_default,
    "XGBoost\ntuned\n+ FE"       : results_fe_tuned,
}
palette = ["#4E79A7", "#F28E2B", "#E15759"]

metrics_cfg = [
    ("pr_auc",  "PR-AUC (primary)"),
    ("roc_auc", "ROC-AUC"),
    ("recall",  "Recall (Diabetes)"),
    ("f1",      "F1 (Diabetes)"),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 6), sharey=False)
x = np.arange(len(model_results))
width = 0.55

for ax, (metric_key, metric_label) in zip(axes, metrics_cfg):
    means = [get_metric(r, metric_key)[0] for r in model_results.values()]
    stds  = [get_metric(r, metric_key)[1] for r in model_results.values()]
    bars  = ax.bar(x, means, width, color=palette, edgecolor="white",
                   alpha=0.90, yerr=stds, capsize=4, error_kw={"lw": 1.5})
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(stds) + 0.008,
                f"{mean:.3f}", ha="center", fontsize=9, fontweight="bold")
    if metric_key == "pr_auc":
        ax.axhline(baseline_rate, color="gray", linestyle="--", lw=1.2,
                   label=f"No-skill ({baseline_rate:.2f})")
        ax.legend(fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(list(model_results.keys()), fontsize=8)
    ax.set_title(metric_label, fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.12)
    sns.despine(ax=ax)

fig.suptitle("Finaler Vergleich: Feature Engineering Effekt auf XGBoost\n"
             "(Stratified 5-Fold CV, mean +/- std)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "24_final_comparison_fe.png"), bbox_inches="tight")
plt.show()

print("\n=== Finaler Vergleich ===")
rows = {}
for name, res in model_results.items():
    lbl = name.replace("\n", " ")
    rows[lbl] = {
        "PR-AUC":     round(get_metric(res, "pr_auc")[0],  4),
        "PR-AUC std": round(get_metric(res, "pr_auc")[1],  4),
        "ROC-AUC":    round(get_metric(res, "roc_auc")[0], 4),
        "Recall":     round(get_metric(res, "recall")[0],  4),
        "F1":         round(get_metric(res, "f1")[0],      4),
    }
print(pd.DataFrame(rows).T.to_string())


In [ ]:
# PR Curves aller drei Modelle
fig, ax = plt.subplots(figsize=(9, 6))
plot_colors = ["#4E79A7", "#F28E2B", "#E15759"]
plot_labels = ["XGBoost tuned (kein FE)", "XGBoost default + FE", "XGBoost tuned + FE"]

for res, color, lbl in zip(model_results.values(), plot_colors, plot_labels):
    mean_pr = np.mean([r["pr_auc"] for r in res])
    for r in res:
        prec, rec, _ = precision_recall_curve(r["y_true"], r["y_prob"])
        ax.plot(rec, prec, alpha=0.35, lw=1.2, color=color)
    ax.plot([], [], lw=2.5, color=color, label=f"{lbl}  (PR-AUC~{mean_pr:.3f})")

ax.axhline(baseline_rate, color="gray", linestyle="--", lw=1.2,
           label=f"No-skill ({baseline_rate:.2f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.set_title("PR Curves - XGBoost Modellvergleich\n"
             "(je 5 Folds pro Modell, transparent)", fontweight="bold")
ax.legend(fontsize=9, loc="upper right")
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "25_pr_curves_comparison.png"), bbox_inches="tight")
plt.show()


## 9 - Zusammenfassung und Interpretation

### Feature Engineering - begruendet durch Notebook 02

Alle 9 neuen Features basieren auf konkreten Erkenntnissen der Datenexploration:
- **BMI_cat**: BMI ist rechtsschiefe kontinuierliche Variable - WHO-Klassen sind klinisch standard
- **Risk_score**: Buendelt die Top-Korrelatioren aus Notebook 02 (HighBP, HighChol, DiffWalk)
- **Any_poor_health**: MentHlth/PhysHlth zero-inflated (75% = 0) - binaeres Flag stabiler als Summe
- **BP_Chol_both**: HighBP x HighChol: moderate Inter-Feature-Korrelation in Notebook 02 bestaetigt
- **Age_GenHlth**: GenHlth (Rang 1) x Age (Rang 4): beide starke Einzelpraediktoren

### Vergleich der drei Modelle

XGBoost tuned (kein FE) vs. XGBoost tuned + FE isoliert den reinen FE-Beitrag.
Die neuen Features helfen, weil sie:
1. Schwache Einzelpraediktoren buendeln (Risk_score, Protective_behaviors)
2. Interaktionen explizit machen, die XGBoost sonst ueber mehrere Splits approximiert (Age_GenHlth)
3. Problematische Verteilungen linearisieren (Any_poor_health statt zero-inflated MentHlth+PhysHlth)

### Was wuerde noch mehr bringen?

- **Bayesian Optimization** (Optuna) statt RandomizedSearchCV
- **Early Stopping** mit separater Validierungsmenge
- **Stacking/Ensemble** mit Random Forest oder LightGBM
- **Threshold-Optimierung** fuer hoehere Recall im Screening-Kontext
- **SHAP-Analyse** fuer Erklaerbarkeit in der Praesentation

-> Naechster Schritt: `05_evaluation.ipynb` - Finale Evaluation, Threshold, SHAP.
